# Lab 7.4 &mdash; Extract: Mess In, Structure Out

**Level:** Beginner &nbsp;|&nbsp; **Est. time:** 25 min &nbsp;|&nbsp; **Day 4 &middot; Module 7 &mdash; Task Automation with AI Agents**

### What you'll do
- Pull defined fields out of an unstructured email
- Use a closed set of intents; handle a missing order id
- Return the tight schema the rest of the pipeline consumes

> **How this lab works (experiential flow):** read the **Concept**, run the **Demo** to see it work, then complete **Your Turn** by replacing every `___` placeholder. Run the **grader** cell at the end &mdash; it prints `[PASS]` / `[FAIL]` / `[TODO]` and a final `Score`. Aim for a full score.

> **Framework note:** the graded steps are **offline &amp; deterministic** (pure Python stdlib); the agent-assembly labs reuse the **LangChain-shaped shim** from Module 6. Advanced labs end with an **optional** cell that runs the **real** library. You are building the **email-drafting agent** &mdash; the client's Lab 4.1.

**Reference:** [Module 7 slides &mdash; Extract — mess in, structure out](../../presentation/day4-module7-task-automation.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 7 labs](./index.html)

In [ ]:
# Setup -- run me first
import os
WORK = "/tmp/biaa-lab-07-04"
os.makedirs(WORK, exist_ok=True)
print("Working dir:", WORK)

## Concept
**Extract** turns unstructured input into structured data (deck slide 10): an email *"my order from
last Tuesday still hasn't arrived, ref 4471, getting frustrated"* becomes
`{"order_id": 4471, "intent": "delivery", "sentiment": "negative"}`. Keys: a **tight schema** (only
the fields you'll use, intents from a **closed set**), **handle missing data** (return `None`, don't
invent an id), and it's usually the **first step** in the chain &mdash; extract &rarr; route &rarr;
draft.

In [ ]:
sample = "Hi, my order from last Tuesday still hasn't arrived, ref 4471, getting frustrated."
print("unstructured in:", sample)
print("we want out    : {order_id, intent, sentiment}")

## Your Turn
Complete `extract`: pull the order id (digits), classify the intent from a closed set, and read a
rough sentiment.

In [ ]:
INTENTS = ("refund", "delivery", "cancel", "other")

def extract(email):
    text = email.lower()
    # order id: the digits in the message, or None if there are none
    digits = "".join(ch for ch in email if ch.isdigit())
    order_id = ___   # TODO: int(digits) if we found any, else None
    # intent: map keywords to a label from the CLOSED set INTENTS
    if "refund" in text or "money back" in text:
        intent = "refund"
    elif ___:   # TODO: a delivery-ish message (deliver / arrive / late / where is)
        intent = "delivery"
    elif "cancel" in text:
        intent = "cancel"
    else:
        intent = "other"
    # sentiment: negative if the customer sounds unhappy
    sentiment = ___   # TODO: "negative" if any unhappy word is present, else "neutral"
    return {"order_id": order_id, "intent": intent, "sentiment": sentiment}

try:
    print(extract("my order 4471 still hasn't arrived, getting frustrated"))
    print(extract("please cancel my subscription"))
    print(extract("I want a refund"))
except Exception as e:
    print('Fill the blanks, then re-run.', type(e).__name__)

In [ ]:
# === Auto-grader: run after filling the blanks above ===
_results = []
def _rec(label, status, extra=""):
    _results.append(status); print(f"[{status}] {label}" + (f" -- {extra}" if extra else ""))
def expect(label, got, want):
    if got == "___" or got is None: _rec(label, "TODO")
    elif got == want: _rec(label, "PASS")
    else: _rec(label, "FAIL", f"got {got!r}")
def expect_true(label, fn):
    try: _rec(label, "PASS" if fn() else "FAIL")
    except Exception as e: _rec(label, "TODO", type(e).__name__)

expect_true("pulls the order id from the text", lambda: extract("ref 4471 please")["order_id"] == 4471)
expect_true("a message with no id -> order_id is None", lambda: extract("where is my stuff?")["order_id"] is None)
expect_true("detects a delivery intent", lambda: extract("my order hasn't arrived, it's late")["intent"] == "delivery")
expect_true("detects a refund intent", lambda: extract("I want a refund")["intent"] == "refund")
expect_true("intent is always from the closed set", lambda: extract("hello there")["intent"] in ("refund", "delivery", "cancel", "other"))
expect_true("reads a negative sentiment", lambda: extract("I am frustrated, still no order")["sentiment"] == "negative")

_p = _results.count("PASS")
print(f"\nScore: {_p}/{len(_results)}")
print("All checks passed -- lab complete!" if _p == len(_results) else "Keep going: fill the blanks marked ___ and re-run.")

---
### Key takeaway
Extract is the workhorse: it turns email, chat and forms into rows your systems can process. A tight schema plus missing-data handling is what makes it reliable enough to build on.

[&#8592; All Module 7 labs](./index.html) &nbsp;&middot;&nbsp; [Module 7 slides](../../presentation/day4-module7-task-automation.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>